# Average Charge Analysis

This notebook streams two legacy covariance-history caches and plots the normalized ensemble drift about half-filling

$$\frac{\left|\mathbb{E}_s\,\mathrm{Tr}\left[\tfrac{1}{2}(G_s + I)\right] - N_x N_y\right|}{N_x N_y}$$

on a log scale versus cycle.

In [ ]:
import os
import re
import sys
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import numpy.lib.format as npformat
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-average-charge")
os.environ.setdefault("XDG_CACHE_HOME", "/tmp")
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
os.chdir(ROOT)

NX = 12
NY = 31
TARGET_CHARGE = NX * NY
TINY = np.finfo(np.float64).tiny

DATASETS = [
    {
        "label": "maxmix random (C50)",
        "path": Path("/home/abhuiyan/class_A_fermionic_adaptive_circuit/cache/G_history_samples/N12x31/N12x31_C50_S100_nshNone_DW1_init-maxmix_n_a0.5_seq-random_exclNone_ps0_pst0_fm0_pc0_markov_circuit_entropy_boundary_timeseries.npz"),
    },
    {
        "label": "default dw_symmetric (C100)",
        "path": Path("/home/abhuiyan/class_A_fermionic_adaptive_circuit/cache/G_history_samples/N12x31/N12x31_C100_S100_nshNone_DW1_init-default_n_a0.5_seq-dw_symmetric_exclNone_ps0_pst0_fm0_pc0_markov_circuit.npz"),
    },
]

plt.style.use("seaborn-v0_8-whitegrid")

## Notes

- Each archive stores a single `G_hist.npy` array with shape `(samples, time, dim, dim)`.
- These arrays are large, so the helper below streams one matrix at a time from the zipped `.npz` archive and only keeps the per-sample charge history.
- The cycle axis is inferred from the filename. If a file says `C100` but stores `51` time slices, the notebook interprets that as cycles `0, 2, ..., 100`.

In [ ]:
def read_npz_array_header(npz_path: Path):
    with zipfile.ZipFile(npz_path) as zf:
        members = zf.namelist()
        if len(members) != 1:
            raise ValueError(f"Expected exactly one member in {npz_path}, found {members}.")
        member = members[0]
        with zf.open(member) as fh:
            version = npformat.read_magic(fh)
            shape, fortran_order, dtype = npformat._read_array_header(fh, version)
    return member, tuple(shape), np.dtype(dtype), bool(fortran_order)


def infer_cycle_axis(npz_path: Path, time_count: int):
    if time_count <= 0:
        raise ValueError("time_count must be positive.")
    if time_count == 1:
        return np.array([0], dtype=np.int64)

    match = re.search(r"_C(\d+)_", npz_path.name)
    if match is None:
        return np.arange(time_count, dtype=np.int64)

    nominal_cycles = int(match.group(1))
    intervals = time_count - 1
    if nominal_cycles == intervals:
        return np.arange(time_count, dtype=np.int64)
    if nominal_cycles % intervals == 0:
        step = nominal_cycles // intervals
        return np.arange(time_count, dtype=np.int64) * step

    return np.linspace(0, nominal_cycles, time_count)


def stream_charge_history(npz_path: Path):
    member, shape, dtype, fortran_order = read_npz_array_header(npz_path)
    if len(shape) != 4 or shape[-1] != shape[-2]:
        raise ValueError(f"Expected rank-4 square-matrix history in {npz_path}, got shape {shape}.")

    sample_count, time_count, dim, _ = shape
    matrix_elems = dim * dim
    matrix_bytes = matrix_elems * dtype.itemsize
    diagonal_stride = dim + 1

    charges = np.empty((sample_count, time_count), dtype=np.float64)

    with zipfile.ZipFile(npz_path) as zf:
        with zf.open(member) as fh:
            version = npformat.read_magic(fh)
            _shape, _fortran_order, _dtype = npformat._read_array_header(fh, version)
            if tuple(_shape) != shape or bool(_fortran_order) != fortran_order or np.dtype(_dtype) != dtype:
                raise RuntimeError(f"Header changed while reopening {npz_path}.")

            for sample_idx in range(sample_count):
                for time_idx in range(time_count):
                    buffer = fh.read(matrix_bytes)
                    if len(buffer) != matrix_bytes:
                        raise EOFError(
                            f"Unexpected end of stream for {npz_path} at sample={sample_idx}, time={time_idx}."
                        )
                    flat = np.frombuffer(buffer, dtype=dtype, count=matrix_elems)
                    trace_val = flat[::diagonal_stride].sum()
                    charges[sample_idx, time_idx] = float(np.real(0.5 * (trace_val + dim)))

    return {
        "member": member,
        "shape": shape,
        "dtype": str(dtype),
        "fortran_order": fortran_order,
        "cycles": infer_cycle_axis(npz_path, time_count),
        "charges": charges,
    }


In [ ]:
results = []
summary_rows = []

for dataset in DATASETS:
    payload = stream_charge_history(dataset["path"])
    charges = payload["charges"]
    ensemble_charge = charges.mean(axis=0)
    ensemble_drift = np.abs(ensemble_charge - TARGET_CHARGE)
    normalized_drift = ensemble_drift / TARGET_CHARGE

    results.append(
        {
            "label": dataset["label"],
            "path": dataset["path"],
            "cycles": np.asarray(payload["cycles"]),
            "ensemble_charge": ensemble_charge,
            "ensemble_drift": ensemble_drift,
            "normalized_drift": normalized_drift,
            "sample_count": charges.shape[0],
            "time_count": charges.shape[1],
            "dim": payload["shape"][-1],
            "member": payload["member"],
        }
    )

    summary_rows.append(
        {
            "label": dataset["label"],
            "samples": charges.shape[0],
            "stored_time_slices": charges.shape[1],
            "matrix_dim": payload["shape"][-1],
            "cycle_start": float(np.asarray(payload["cycles"])[0]),
            "cycle_end": float(np.asarray(payload["cycles"])[-1]),
            "ensemble_charge_start": float(ensemble_charge[0]),
            "ensemble_charge_end": float(ensemble_charge[-1]),
            "normalized_drift_start": float(normalized_drift[0]),
            "normalized_drift_end": float(normalized_drift[-1]),
            "path": str(dataset["path"]),
        }
    )

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

fig, ax = plt.subplots(figsize=(9, 5.5), constrained_layout=True)
for result in results:
    y = np.maximum(result["normalized_drift"], TINY)
    ax.plot(result["cycles"], y, marker="o", ms=3, lw=1.6, label=result["label"])

ax.set_yscale("log")
ax.set_xlabel("cycle")
ax.set_ylabel(r"$|\mathbb{E}_s Q_s - N_x N_y| / (N_x N_y)$")
ax.set_title("Normalized ensemble drift about half-filling vs cycle")
ax.legend(frameon=True)
ax.grid(True, which="both", linestyle=":", linewidth=0.8)
plt.show()

In [ ]:
from fgtn.classA_U1FGTN import classA_U1FGTN

POSTSELECT_CYCLES = 50
POSTSELECT_INIT_MODE = "default"
POSTSELECT_SEQUENCE = "random"

post_model = classA_U1FGTN(
    Nx=NX,
    Ny=NY,
    DW=True,
    nshell=None,
    filling_frac=0.5,
    alpha_1=1.0,
    alpha_2=30.0,
    trial_orbitals="X",
    dw_truncation=False,
)
post_model.construct_OW_projectors(nshell=None, DW=True, trial_orbitals="X", dw_truncation=False)

post_result = post_model.run_markov_circuit(
    G_history=True,
    progress=False,
    cycles=POSTSELECT_CYCLES,
    postselect=True,
    samples=1,
    init_mode=POSTSELECT_INIT_MODE,
    save=False,
    sequence=POSTSELECT_SEQUENCE,
)

post_hist = np.asarray(post_result["G_hist"][0], dtype=np.complex128)
post_dim = post_hist.shape[-1]
post_cycles = np.arange(post_hist.shape[0], dtype=np.int64)
post_charges = 0.5 * (np.trace(post_hist, axis1=1, axis2=2).real + post_dim)
post_normalized_drift = np.abs(post_charges - TARGET_CHARGE) / TARGET_CHARGE

display(pd.DataFrame({
    "cycle": post_cycles,
    "ensemble_charge": post_charges,
    "normalized_drift": post_normalized_drift,
}).head())

fig, ax = plt.subplots(figsize=(8.5, 5.0), constrained_layout=True)
ax.plot(
    post_cycles,
    np.maximum(post_normalized_drift, TINY),
    marker="o",
    ms=3,
    lw=1.5,
    color="black",
    label=f"postselect {POSTSELECT_INIT_MODE}, {POSTSELECT_SEQUENCE}, S=1",
)
ax.set_yscale("log")
ax.set_xlabel("cycle")
ax.set_ylabel(r"$|Q - N_x N_y| / (N_x N_y)$")
ax.set_title("Single-sample postselected normalized half-filling drift")
ax.legend(frameon=True)
ax.grid(True, which="both", linestyle=":", linewidth=0.8)
plt.show()